# Setup

In [ ]:
from lets_plot import *
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

LetsPlot.setup_html()

In [43]:
# BASE_URL = "https://github.com/byuidatascience/data4dwellings/raw/master/data-raw/dwellings_ml/dwellings_ml.csv"
# OPT_URL  = "https://github.com/byuidatascience/data4dwellings/raw/master/data-raw/dwellings_neighborhoods_ml/dwellings_neighborhoods_ml.csv"
# INFO_URL = "https://github.com/byuidatascience/data4dwellings/raw/master/data-raw/dwellings_denver/dwellings_denver.csv"

# base_df = pd.read_csv(BASE_URL)
# opt_df = pd.read_csv(OPT_URL)
# info_df = pd.read_csv(INFO_URL)

BASE_CSV = "dwellings_ml.csv"
OPT_CSV  = "dwellings_neighborhoods_ml.csv"
INFO_CSV = "dwellings_denver.csv"

df = pd.read_csv(BASE_CSV)
opt_df = pd.read_csv(OPT_CSV)
info_df = pd.read_csv(INFO_CSV)

In [47]:
data_dict_preparsed = \
"""
|parcel                           |character |The parcel id                                 |
|abstrprd                         |numeric   |No clue                                       |
|livearea                         |numeric   |Square footage that is liveable               |
|finbsmnt                         |numeric   |Square footage finished in the basement       |
|basement                         |numeric   |Total square footage of the basement          |
|yrbuilt                          |numeric   |Year the home was built                       |
|totunits                         |numeric   |How many dwelling units in the building       |
|stories                          |numeric   |The number of stories                         |
|nocars                           |numeric   |size of the garage in cars                    |
|numbdrm                          |numeric   |Number of bedrooms                            |
|numbaths                         |numeric   |Number of bathrooms                           |
|sprice                           |numeric   |Selling price                                 |
|deduct                           |numeric   |Deduction from the selling price              |
|netprice                         |numeric   |Net price of home                             |
|tasp                             |numeric   |Tax assesed selling price                     |
|smonth                           |numeric   |Month sold                                    |
|syear                            |numeric   |Year sold                                     |
"""

lines_preparsed = [line.strip('|').strip() for line in data_dict_preparsed.split('\n') if line.strip()]
mapped_lines = [tuple(line.split('|')[0::2]) for line in lines_preparsed]
data_dict = dict((line[0].strip(), line[1].strip()) for line in mapped_lines)

In [39]:
data_dict

{'parcel': 'The parcel id',
 'abstrprd': 'No clue',
 'livearea': 'Square footage that is liveable',
 'finbsmnt': 'Square footage finished in the basement',
 'basement': 'Total square footage of the basement',
 'yrbuilt': 'Year the home was built',
 'totunits': 'How many dwelling units in the building',
 'stories': 'The number of stories',
 'nocars': 'size of the garage in cars',
 'numbdrm': 'Number of bedrooms',
 'numbaths': 'Number of bathrooms',
 'sprice': 'Selling price',
 'deduct': 'Deduction from the selling price',
 'netprice': 'Net price of home',
 'tasp': 'Tax assesed selling price',
 'smonth': 'Month sold',
 'syear': 'Year sold'}

# Q1

In [ ]:
df["before1980"] = df["before1980"].astype(bool)
df["era"] = df["before1980"].map({True: "Before 1980", False: "1980 or Later"})

plot_data = df[["livearea", "nocars", "numbdrm", "era"]]

In [ ]:
plot1 = (
    ggplot(plot_data, aes(x=as_discrete("era"), y="livearea", fill="era")) +
    geom_boxplot() +
    ggtitle("Living Area by Era") +
    xlab("Built Era") +
    ylab("Living Area (sq ft)")
)
plot1.show()

In [ ]:
plot2 = (
    ggplot(plot_data, aes(x=as_discrete("era"), y="nocars", fill="era")) +
    geom_boxplot() +
    ggtitle("Number of Cars by Era") +
    xlab("Built Era") +
    ylab("Garage Capacity")
)
plot2.show()

In [6]:
plot3 = (
    ggplot(plot_data, aes(x=as_discrete("era"), y="numbdrm", fill="era")) +
    geom_boxplot() +
    ggtitle("Bedrooms by Era") +
    xlab("Built Era") +
    ylab("Number of Bedrooms")
)
plot3.show()

# Q2

In [ ]:
model_df = df.copy()
X = model_df.drop(columns=["parcel", "before1980", "yrbuilt"])
y = model_df["before1980"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [16]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred)
# rf_rep = classification_report(y_test, y_pred, output_dict=True)

In [19]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)

c:\Program Files\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_acc = accuracy_score(y_test, gb_pred)

In [22]:
print(f"Random Forest: {rf_acc}\nLogistic Regression: {lr_acc}\nGradient Boosting: {gb_acc}")

Random Forest: 0.9125027274710888
Logistic Regression: 0.8306785948068951
Gradient Boosting: 0.9319223216233908


# Q3

In [48]:
importance_df = (
    pd.DataFrame({"Feature": X.columns, "Importance": gb.feature_importances_})
    .sort_values("Importance", ascending=False)
    .head(15)
)

importance_df["Label"] = importance_df["Feature"].map(data_dict).fillna(importance_df["Feature"])

importance_df["Label"] = pd.Categorical(
    importance_df["Label"],
    categories=importance_df.sort_values("Importance")["Label"],
    ordered=True
)

plot = (
    ggplot(importance_df, aes(x="Label", y="Importance")) +
    geom_bar(stat="identity", fill="#4C72B0") +
    coord_flip() +
    ggtitle("Top 15 Label Importances - Gradient Boosting") +
    xlab("Label") +
    ylab("Importance Score")
)
plot.show()

# Q4

In [ ]:
gb_accuracy = accuracy_score(y_test, gb_pred)
gb_precision = precision_score(y_test, gb_pred)
gb_recall = recall_score(y_test, gb_pred)
gb_f1 = f1_score(y_test, gb_pred)

print(f"""Gradient Boosted Regression:
Accuracy: {gb_accuracy}
Precision: {gb_precision}
Recall: {gb_recall}
F1 Score: {gb_f1}""")


Gradient Boosted Regression:
Accuracy: 0.9319223216233908
Precision: 0.9452198185624564
Recall: 0.9458798882681564
F1 Score: 0.9455497382198953
